In [12]:
# =============================================================================
# 02_ingestion: Bronze layer ingestion from the Blue Owls Data API
# -----------------------------------------------------------------------------
# What this notebook does:
#   1. Logs in to the API and refreshes the token automatically on 401
#   2. Retries on 429 / 500 / network errors using exponential backoff
#   3. Pulls every endpoint page by page
#   4. Validates each record; bad records go to a quarantine file, not Bronze
#   5. Appends good records to Bronze CSVs (append-only)
#   6. Tracks finished pages in a manifest file, so re-runs add no duplicates
# =============================================================================

import os, json, time, random, logging
from datetime import datetime, timezone
import requests
import pandas as pd

# ---- Connection settings ----------------------------------------------------
# Read from environment variables first (set by docker-compose), with defaults
# so the notebook also works without them. No secrets are hard-coded in Git.
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

# ---- Output folders ----------------------------------------------------------
# /home/jovyan/work is the same folder as "submission/" on the host machine.
OUTPUT_DIR     = "/home/jovyan/work/output"
BRONZE_DIR     = f"{OUTPUT_DIR}/bronze"       # raw data, one CSV per endpoint
QUARANTINE_DIR = f"{OUTPUT_DIR}/quarantine"   # malformed records, kept for review
LOG_DIR        = f"{OUTPUT_DIR}/logs"         # ingestion log file
MANIFEST_PATH  = f"{OUTPUT_DIR}/manifest.json"  # remembers which pages are done
for d in (BRONZE_DIR, QUARANTINE_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)             # create folders if missing

# ---- Pipeline settings -------------------------------------------------------
DATE_FROM       = "2018-07-01"  # data scope given in the assignment
PAGE_SIZE       = 5000          # API maximum; fewer requests means fewer chances to fail
MAX_RETRIES     = 8             # max attempts for one request before giving up
REQUEST_TIMEOUT = 30            # seconds to wait for a response before treating it as failed

# ---- Endpoint configuration --------------------------------------------------
# key         : columns that uniquely identify a record (used for validation)
# date_filter : True if the API supports ?date_from= for this endpoint
# columns     : the columns we expect; anything missing marks a record as bad
ENDPOINTS = {
    "orders": {
        "key": ["order_id"], "date_filter": True,
        "columns": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                    "order_approved_at", "order_delivered_carrier_date",
                    "order_delivered_customer_date", "order_estimated_delivery_date"]},
    "order_items": {
        "key": ["order_id", "order_item_id"], "date_filter": True,
        "columns": ["order_id", "order_item_id", "product_id", "seller_id",
                    "shipping_limit_date", "price", "freight_value"]},
    "customers": {
        "key": ["customer_id"], "date_filter": False,
        "columns": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                    "customer_city", "customer_state"]},
    "products": {
        "key": ["product_id"], "date_filter": False,
        # Note: "lenght" is the API's own spelling (a typo in the original dataset).
        # We keep it as-is in Bronze and fix the names in Silver.
        "columns": ["product_id", "product_category_name", "product_name_lenght",
                    "product_description_lenght", "product_photos_qty", "product_weight_g",
                    "product_length_cm", "product_height_cm", "product_width_cm"]},
    "sellers": {
        "key": ["seller_id"], "date_filter": False,
        "columns": ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]},
    "payments": {
        "key": ["order_id", "payment_sequential"], "date_filter": False,
        "columns": ["order_id", "payment_sequential", "payment_type",
                    "payment_installments", "payment_value"]},
}

# ---- Logging -----------------------------------------------------------------
# Messages go to the screen and to a file, so a failed run can be investigated later.
logger = logging.getLogger("ingestion")
logger.setLevel(logging.INFO)
logger.handlers.clear()   # avoids duplicate log lines if this cell is run twice
fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")
for h in (logging.StreamHandler(), logging.FileHandler(f"{LOG_DIR}/ingestion.log")):
    h.setFormatter(fmt)
    logger.addHandler(h)

print("Settings ready")

Settings ready


In [3]:
class ApiError(Exception):
    """Raised when a request cannot be completed, even after all retries."""
    pass


def backoff_seconds(attempt):
    """
    Exponential backoff: how long to wait before the next retry.
    attempt 1 -> ~1s, 2 -> ~2s, 3 -> ~4s, 4 -> ~8s ... capped at 30s.
    A random 'jitter' (0-1s) is added so retries don't all hit the server together.
    """
    return min(30, 2 ** (attempt - 1)) + random.uniform(0, 1)


class ApiClient:
    """
    Wraps all HTTP calls so the rest of the notebook never has to think about
    authentication or transient failures.
    """

    def __init__(self):
        self.session = requests.Session()  # reuses the connection, faster than plain requests.get
        self.token = None                  # filled in by login()

    def login(self):
        """Get a fresh Bearer token. The auth endpoint is never failure-injected,
        so only network problems are retried here; wrong credentials fail immediately."""
        for attempt in range(1, 4):
            try:
                r = self.session.post(f"{API_BASE_URL}/api/v1/auth/token",
                                      json={"username": API_USERNAME, "password": API_PASSWORD},
                                      timeout=REQUEST_TIMEOUT)
            except (requests.ConnectionError, requests.Timeout) as e:
                logger.warning(f"Login network error ({e}); retry {attempt}/3")
                time.sleep(backoff_seconds(attempt))
                continue
            if r.status_code == 200:
                self.token = r.json()["access_token"]
                logger.info("Login OK, new token received")
                return
            # Anything other than 200 here (e.g. 401 bad password) will not fix itself.
            raise ApiError(f"Login failed: {r.status_code} {r.text[:200]}")
        raise ApiError("Login failed after 3 tries")

    def get(self, path, params):
        """
        GET with automatic handling of every failure the API is known to inject:
          200 -> return the parsed JSON
          401 -> token expired: log in again, then retry
          429 -> rate limited: wait for the Retry-After header, then retry
          5xx -> server error: wait with exponential backoff, then retry
          network error / bad JSON -> treated like a 5xx
          other 4xx -> a real bug in our request, so fail immediately (no pointless retry)
        """
        if self.token is None:
            self.login()
        url = f"{API_BASE_URL}{path}"

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                r = self.session.get(url, params=params, timeout=REQUEST_TIMEOUT,
                                     headers={"Authorization": f"Bearer {self.token}"})
            except (requests.ConnectionError, requests.Timeout) as e:
                wait = backoff_seconds(attempt)
                logger.warning(f"Network error ({e}); waiting {wait:.1f}s, attempt {attempt}/{MAX_RETRIES}")
                time.sleep(wait)
                continue

            if r.status_code == 200:
                try:
                    return r.json()
                except ValueError:
                    # A 200 response with a broken body: retry instead of crashing.
                    logger.warning(f"Response is not valid JSON; retry {attempt}/{MAX_RETRIES}")
                    time.sleep(backoff_seconds(attempt))
                    continue
            elif r.status_code == 401:
                # Tokens expire (15 min). Get a new one and loop to retry the same request.
                logger.warning("401: token expired, logging in again")
                self.login()
            elif r.status_code == 429:
                # The API tells us how long to wait; respect it, plus a small jitter.
                wait = float(r.headers.get("Retry-After", 2)) + random.uniform(0, 1)
                logger.warning(f"429 rate limit; waiting {wait:.1f}s, attempt {attempt}/{MAX_RETRIES}")
                time.sleep(wait)
            elif r.status_code >= 500:
                wait = backoff_seconds(attempt)
                logger.warning(f"{r.status_code} server error; waiting {wait:.1f}s, attempt {attempt}/{MAX_RETRIES}")
                time.sleep(wait)
            else:
                raise ApiError(f"Non-retryable status {r.status_code}: {r.text[:200]}")

        # All attempts used up. Raise a controlled error so the caller can decide what to do.
        raise ApiError(f"Failed after {MAX_RETRIES} attempts: {path} {params}")

print("API client ready")

API client ready


In [4]:
# -----------------------------------------------------------------------------
# MANIFEST: a small JSON file that remembers which pages were ingested.
# This is how the pipeline stays idempotent: finished pages are never fetched again.
# -----------------------------------------------------------------------------
def load_manifest():
    """Read the manifest if it exists, else start with an empty one (first run)."""
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    return {"completed_pages": {}}


def save_manifest(manifest):
    """
    Save atomically: write to a temp file first, then rename it over the real file.
    If the notebook crashes mid-write, the old manifest stays intact and is not
    left half-written (corrupted).
    """
    tmp = MANIFEST_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp, MANIFEST_PATH)


# -----------------------------------------------------------------------------
# VALIDATION: decide whether a record is usable.
# Returns None when the record is fine, or a short reason string when it is bad.
# -----------------------------------------------------------------------------
def validate_record(rec, endpoint):
    cfg = ENDPOINTS[endpoint]

    # Must be a JSON object (dict), not a string, number, list, etc.
    if not isinstance(rec, dict):
        return "record is not an object"

    # All expected columns must be present.
    missing = [c for c in cfg["columns"] if c not in rec]
    if missing:
        return f"missing columns: {missing}"

    # Key fields (like order_id) must have a value, or we can't identify the row.
    for k in cfg["key"]:
        if rec[k] is None or str(rec[k]).strip() == "":
            return f"key field is empty: {k}"

    # Nested objects cannot be stored in a flat CSV.
    if any(isinstance(v, (dict, list)) for v in rec.values()):
        return "nested value found"

    return None


# -----------------------------------------------------------------------------
# WRITING FILES
# -----------------------------------------------------------------------------
def write_bronze(endpoint, good_records, ingested_at):
    """
    Append valid records to the Bronze CSV for this endpoint.
    Values are stored exactly as the API sent them (all text); typing happens in Silver.
    Two metadata columns are added as required: _ingested_at and _source_endpoint.
    """
    cols = ENDPOINTS[endpoint]["columns"]
    df = pd.DataFrame([{c: r[c] for c in cols} for r in good_records], columns=cols)
    df["_ingested_at"] = ingested_at
    df["_source_endpoint"] = endpoint
    path = f"{BRONZE_DIR}/{endpoint}.csv"
    # mode="a" appends (Bronze is append-only); header is written only for a new file.
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def write_quarantine(endpoint, page, bad_records):
    """
    Save bad records with the reason and page number, one JSON object per line.
    Nothing is silently dropped: every rejected record can be reviewed here.
    """
    path = f"{QUARANTINE_DIR}/{endpoint}_quarantine.jsonl"
    now = datetime.now(timezone.utc).isoformat()
    with open(path, "a") as f:
        for rec, reason in bad_records:
            f.write(json.dumps({"endpoint": endpoint, "page": page, "reason": reason,
                                "quarantined_at": now, "record": rec}, default=str) + "\n")

print("Helpers ready")

Helpers ready


In [5]:
def ingest_endpoint(client, manifest, endpoint):
    """
    Pull every page of one endpoint and write it to Bronze.
    Returns a small dict of counters for the final summary.
    """
    cfg = ENDPOINTS[endpoint]
    # Part of the manifest key. If DATE_FROM changes, pages are treated as new,
    # because a different date range is a different extraction.
    date_tag = DATE_FROM if cfg["date_filter"] else "none"
    stats = {"pages_fetched": 0, "pages_skipped": 0, "rows_written": 0,
             "rows_quarantined": 0, "failed_pages": 0}
    page, consecutive_failures = 1, 0

    while True:
        # Unique id of this exact request. Same id on a re-run means "already done".
        key = f"{endpoint}|date_from={date_tag}|page_size={PAGE_SIZE}|page={page}"

        # ---- Idempotency check: skip pages finished in an earlier run ----------
        if key in manifest["completed_pages"]:
            stats["pages_skipped"] += 1
            # We still need to know if a next page exists, so it is stored in the manifest.
            if not manifest["completed_pages"][key]["has_next"]:
                break
            page += 1
            continue

        # ---- Build request parameters ------------------------------------------
        params = {"page": page, "page_size": PAGE_SIZE}
        if cfg["date_filter"]:
            params["date_from"] = DATE_FROM  # only orders/order_items support this

        # ---- Fetch the page (retries happen inside client.get) -----------------
        try:
            body = client.get(f"/api/v1/data/{endpoint}", params)
            # Check the shape of the response before trusting it.
            if (not isinstance(body, dict) or not isinstance(body.get("data"), list)
                    or not isinstance(body.get("pagination"), dict)):
                raise ApiError("unexpected response structure")
        except ApiError as e:
            # This page failed even after all retries. Log it and move on, rather than
            # crash the whole pipeline. It is NOT written to the manifest, so the next
            # run will try this page again.
            logger.error(f"{endpoint} page {page} failed: {e}")
            stats["failed_pages"] += 1
            consecutive_failures += 1
            if consecutive_failures >= 3:
                # Probably a real outage. Stop instead of hammering the API.
                logger.error(f"{endpoint}: 3 pages failed in a row, stopping. Re-run to resume from here.")
                break
            page += 1
            continue
        consecutive_failures = 0

        records = body["data"]
        has_next = bool(body["pagination"].get("has_next", False))

        # ---- Validate each record: good ones to Bronze, bad ones to quarantine -
        good, bad = [], []
        for rec in records:
            reason = validate_record(rec, endpoint)
            if reason is None:
                good.append(rec)
            else:
                bad.append((rec, reason))

        ingested_at = datetime.now(timezone.utc).isoformat()
        if good:
            write_bronze(endpoint, good, ingested_at)
        if bad:
            write_quarantine(endpoint, page, bad)
            logger.warning(f"{endpoint} page {page}: {len(bad)} bad records sent to quarantine")

        # ---- Mark the page as done, but only AFTER the data is safely written --
        # If the notebook crashes before this line, the page is retried next run.
        # (Worst case is a repeated page, never a lost page.)
        manifest["completed_pages"][key] = {"rows": len(good), "quarantined": len(bad),
                                            "has_next": has_next, "ingested_at": ingested_at}
        save_manifest(manifest)

        stats["pages_fetched"] += 1
        stats["rows_written"] += len(good)
        stats["rows_quarantined"] += len(bad)
        logger.info(f"{endpoint} page {page}: {len(good)} rows OK")

        # ---- Stop when the API says there are no more pages --------------------
        if not has_next or not records:
            break
        page += 1

    return stats

print("Ingestion function ready")

Ingestion function ready


In [11]:
client = ApiClient()
manifest = load_manifest()   # empty on the first run, populated on re-runs
all_stats = {}

for endpoint in ENDPOINTS:
    logger.info(f"===== {endpoint} started =====")
    try:
        all_stats[endpoint] = ingest_endpoint(client, manifest, endpoint)
    except Exception as e:
        # Safety net: one endpoint failing unexpectedly must not stop the others.
        logger.error(f"{endpoint}: unexpected error: {e}")
        all_stats[endpoint] = {"error": str(e)}

print("\n===== SUMMARY =====")
print(pd.DataFrame(all_stats).T)

14:02:03 [INFO] ===== orders shuru =====
14:02:03 [INFO] ===== order_items shuru =====
14:02:03 [INFO] ===== customers shuru =====
14:02:03 [INFO] ===== products shuru =====
14:02:03 [INFO] ===== sellers shuru =====
14:02:03 [INFO] ===== payments shuru =====



===== SUMMARY =====
             pages_fetched  pages_skipped  rows_written  rows_quarantined  \
orders                   0              3             0                 0   
order_items              0              4             0                 0   
customers                0             20             0                 0   
products                 0              7             0                 0   
sellers                  0              1             0                 0   
payments                 0             21             0                 0   

             failed_pages  
orders                  0  
order_items             0  
customers               0  
products                0  
sellers                 0  
payments                0  


In [7]:
# Sanity check: row count per file, and duplicates on each table's natural key.
# duplicate_keys must be 0, otherwise the idempotency logic is not working.
for ep, cfg in ENDPOINTS.items():
    path = f"{BRONZE_DIR}/{ep}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path, dtype=str)
        dups = df.duplicated(subset=cfg["key"]).sum()
        print(f"{ep:12s} rows={len(df):>7}  duplicate_keys={dups}")
    else:
        print(f"{ep:12s} file not created")

orders       rows=  12824  duplicate_keys=0
order_items  rows=  15590  duplicate_keys=0
customers    rows=  99441  duplicate_keys=0
products     rows=  32951  duplicate_keys=0
sellers      rows=   3095  duplicate_keys=0
payments     rows= 103886  duplicate_keys=0
